# 📉 05-学习率调度与训练策略

> 目标：warmup / cosine / step 调度的**为什么**，以及梯度裁剪、梯度累积、EMA 三个训练技巧的数学原理。


## 核心概念

### (a) 🗂️ 学习率调度与训练策略知识地图

这篇讲学习率不能一条直线走到底：前期要 warmup 小步预热、后期要衰减收尾，并配套梯度裁剪、梯度累积、EMA/SWA 等训练稳定技巧，全是工程实操。

```mermaid
flowchart TB
    Node1["🧠 为什么需要调度"]
    Node1 --> Node2["🧠 warmup 预热"]
    Node2 --> Node3["🧠 cosine 余弦衰减"]
    Node3 --> Node4["🧠 one-cycle 单周期"]
    Node4 --> Node5["🧠 调度 × 优化器"]
    Node5 --> Node6["🧠 梯度裁剪"]
    Node6 --> Node7["🧠 梯度累积"]
    Node7 --> Node8["🧠 EMA / SWA"]
    Node8 --> Node9["🧠 公式小抄"]
```

- **节点1（为什么需要调度）**：前期模型参数还很"生"，大步长会把训练带飞（尤其 Adam 二阶矩冷启动时），后期梯度变小需要更细的步子才能磨到谷底，所以学习率要按阶段变。
- **节点2（warmup 预热）**：前 N 步从很小的 lr 线性升到峰值，让优化器状态（如 Adam 的 m/v）先"热起来"再全速前进；大模型训练标配（如 GPT 用 2000 步 warmup）。
- **节点3（cosine 余弦衰减）**：学习率按余弦曲线从峰值平滑降到接近 0，后期"慢慢磨"而不突变；手算例子能看到 100 步内的具体取值，是预训练/微调的主流收尾方式。
- **节点4（one-cycle 单周期）**：先升后降的"倒 U"策略：前半段升温快速探索、后半段降温精细收敛，小数据集上常能更快达到好效果。
- **节点5（调度 × 优化器）**：SGD 配调度收益最大（它靠步长控制一切），Adam/AdamW 也要 warmup+decay 但峰值 lr 更小；实验证明调度能把"震荡"换成"收敛"。
- **节点6（梯度裁剪）**：当梯度范数超阈值就整体缩放到阈值，方向不变只限步长，专治梯度爆炸；阈值 1.0 只是起点，按范数量级调。
- **节点7（梯度累积）**：显存不够时把 N 个小 batch 的梯度累加再更新，等价于大 batch；三个隐蔽细节：loss 要除以 N、BN 统计量处理、梯度尺度。
- **节点8（EMA / SWA）**：对参数做指数滑动平均（EMA）或对多个快照求平均（SWA），得到"更钝更稳"的模型，测试精度常比最终快照更好。
- **节点9（公式小抄）**：线性/step/cosine 三种调度一张表 + warmup 手算表，考前默写用；面试常问"warmup 为什么必要""cosine 为什么平滑"。

## 📌 本节路线图

本节分两大部分：**学习率调度**（§1）与**三个训练技巧**（§2 梯度裁剪、§3 梯度累积、§4 EMA/SWA）。调度部分先讲「为什么」（§1.1～§1.3：历史背景与两条直觉），再讲「怎么做」（§1.4～§1.5：公式逐步推导 + 手算例子），最后讲「有什么坑」（§1.6～§1.8：one-cycle 延伸、局限与补救、与优化器的搭配）。

学习建议：每个实验代码都可直接运行，强烈建议改参数观察——例如把 $T_w$ 从 200 改成 20、把峰值从 $10^{-3}$ 改成 $10^{-2}$，体会曲线形状与收敛结果的差异。

符号约定：$\eta$ 学习率、$\eta_{max}$ 峰值、$\eta_{min}$ 下限、$T_w$ warmup 步数、$T$ 总步数、$\|g\|_2$ 梯度 L2 范数。

## §1 为什么需要调度

- **前期（warmup）**：训练初期梯度方差大、方向不可信，直接大步长会破坏预训练分布 / 让 loss 炸 → 从小步长线性升到峰值
- **后期（decay）**：接近极小值时步长应缩小才能精细收敛（振荡减小）
- **常用组合**：`linear warmup -> cosine decay`（LLM 标配）、`step decay`（每隔 N 轮 ×0.1）、`exp decay`、`plateau`（loss 停降再减）

$$\text{warmup: } \eta_t = \eta_{max}\frac{t}{T_w}\ (t<T_w)\qquad\text{cosine: } \eta_t = \eta_{min} + \frac{\eta_{max}-\eta_{min}}{2}\left(1+\cos\frac{\pi(t-T_w)}{T-T_w}\right)$$

## §1.1 历史背景：warmup 与 cosine 从哪里来

warmup 进入主流，要归功于**大批量训练**。Goyal 等人在 2017 年提出**线性缩放规则**：把 batch 从 $B$ 放大到 $kB$ 时，学习率也应乘以 $k$，才能保持相似的训练动态。然而直接按规则放大 lr，训练**前几步**就会不稳定——梯度方向噪声极大、batch 统计量尚未建立，大步长等于蒙眼狂奔。

Goyal 等人的补救正是 warmup：前若干个 epoch（论文用 5 个）把 lr 从很小的值**线性升到目标值**，等优化过程平稳后再放开。从此「大批量 + 线性缩放 + warmup」成为可复现训练的标准配方，也解释了为什么 warmup 步数必须随 batch 放大而加长。

cosine annealing 来自 Loshchilov 与 Hutter 的 SGDR 论文（ICLR 2017）：与其让 lr 阶梯式突降，不如让它在 $\eta_{max}\to\eta_{min}$ 之间按余弦函数**平滑衰减**，周期结束还可以 warm restart 重新升回峰值。后期精细收敛、前期保留探索，正是大模型预训练最需要的节奏——GPT / LLaMA 系列如今标配「linear warmup + cosine decay」。

## §1.2 直觉再深化①：前期为什么步长必须小

除大批量场景外，**小步长起步对 Adam 类优化器尤其重要**。Adam 维护一阶矩 $m_t$ 与二阶矩 $v_t$，二者从 0 初始化：若不做偏差校正，$v_t$ 在早期被低估为真实值的约 $1-\beta_2^t$（$\beta_2=0.999$ 时 $t=10$ 只有约 $1\%$），而有效步长 $\alpha\,\hat m_t/(\sqrt{\hat v_t}+\epsilon)$ 的分母被低估 → 步长被放大。

即便 PyTorch 默认开启偏差校正，**前几步的统计量也只由极少量样本估计出来，方差大、方向不可信**。噪声大的方向配大步长，等于在高维曲面上盲冲：loss 可能 spike，甚至一步把预训练好的参数推出分布。warmup 相当于先把开关调小，等 $m_t,v_t$ 被足够多真实梯度「焐热」、方向可信后再放开到峰值。

另一个视角是**初始化几何**：随机初始化后损失曲面早期陡峭、等高线病态，小步长先建立正确的局部感知，再逐步提速。LLM 预训练的 warmup 常占总步数 $1\%\sim5\%$（500B token 的训练对应上千步），就是为让几十亿参数平稳起步。

## §1.3 直觉再深化②：后期为什么要衰减

训练后期 loss 进入「盆地」，常数 lr 会让参数在极小值附近来回震荡——**步长大于盆地宽度时，最优解会被反复跳过**。实验 2 会看到：同样的目标函数，常数 lr 的最终 $|w|$ 明显大于衰减调度。衰减的本质，是把有效步长压缩到小于盆地的特征尺度，让更新精细地落向极小点。

衰减还与**泛化**有关：小的有效步长让优化器更容易停在大而平坦的极小（flat minima），而平坦区域通常对应更好的测试表现。这正是 EMA / SWA（§4）想人为复制的状态——把最后几个权重平均，等效于「住在盆地底部」而不是「在盆地边缘晃」。

一句话总结调度哲学：**前期要敢探索但别失控，后期要收得拢、落得准**。warmup 管前期、decay 管后期，二者互补，所以大模型标配「warmup + cosine」。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# ---------- 实验 1：四种调度曲线 ----------
def schedule(kind, t, T=2000, Tw=200, peak=1e-3):
    if kind == 'warmup_cosine':
        if t < Tw:
            return peak * t / Tw
        return peak * 0.5 * (1 + np.cos(np.pi * (t - Tw) / (T - Tw)))
    if kind == 'step':
        return peak * 0.1 ** (t // 400)
    if kind == 'exp':
        return peak * 0.998 ** t
    if kind == 'constant':
        return peak

ts = np.arange(2000)
fig, ax = plt.subplots(figsize=(8, 4.0))
for kind, col, lab in [('warmup_cosine', '#4C72B0', 'warmup + cosine'),
                       ('step', '#55A868', 'step decay (×0.1/400轮)'),
                       ('exp', '#DD8452', 'exp decay'),
                       ('constant', '#C44E52', 'constant')]:
    ax.plot(ts, [schedule(kind, t) for t in ts], label=lab, color=col, lw=1.5)
ax.set_xlabel('step'); ax.set_ylabel('learning rate')
ax.set_title('常见学习率调度', fontsize=12)
ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

### 实验 1 复盘：四条曲线怎么读

横轴是优化器 step，纵轴是 lr。**warmup + cosine（蓝）**先线性爬升到峰值、再平滑衰减——前期给统计量留时间，后期精细收敛，是本文主角。**step（绿）**每 400 步乘 0.1、阶梯式突降——简单可预期，但突降瞬间 lr 跳变，可能扰动已收敛的结构。**exp（橙）**每步乘 0.998、指数式缓降——衰减太慢时后期几乎不动，近似一个更小的常数 lr。**constant（红）**全程不变——作为对照组，用来量化「衰减到底带来多少收益」。

动手实验：把 $T_w$ 改成 20（更陡的爬升）重跑，前几步曲线会明显更「冲」；这就是 §1.7 要讲的 warmup 步数敏感性。

![图 1：线性 warmup + cosine 衰减的学习率曲线示意](images/opt05_lr.png)

**图 1（Wikimedia Commons，CC0 公有领域）** 是典型的「linear warmup + cosine decay」学习率曲线：学习率先以恒定斜率从 0 爬升到峰值，再沿半个余弦周期平滑回落，最终停在略高于 0 的 $\eta_{min}$ 附近——与我们实验 1 的蓝色曲线形状完全一致：warmup 段对应「统计量焐热期」，cosine 段对应「精细收敛期」。

请注意 cosine 的**两端平、中间陡**：峰值附近斜率接近 0、改动极小、保留探索；越过 $\pi/2$ 后快速下降，把大部分衰减倾斜用在中段；接近 $\eta_{min}$ 时再次放缓，避免收尾过冲。

这也是 cosine 比 step decay 更受 LLM 欢迎的原因：曲线处处连续、没有突变，任何一步都不会因 lr 骤降而破坏已经收敛的结构。

## §1.4 公式逐步推导：三类调度

**① 线性 warmup**。设峰值 $\eta_{max}$、warmup 步数 $T_w$，第 $t$ 步（$0\le t\le T_w$）
$$\eta_t=\eta_{max}\cdot\frac{t}{T_w}$$
从 0 到 $\eta_{max}$ 等差爬升：$t=0$ 时 $\eta=0$（第一步无更新），$t=T_w$ 时恰好到峰值。实现上常写成 $\eta_{max}\cdot\min(1,\ t/T_w)$，与后续调度拼成一段分段函数。

**② cosine decay**。在 $[T_w,T]$ 区间内设相位 $\phi=\pi(t-T_w)/(T-T_w)$，则
$$\eta_t=\eta_{min}+\frac{\eta_{max}-\eta_{min}}{2}\bigl(1+\cos\phi\bigr)$$
相位从 0 走到 $\pi$：$\cos 0=1$ 给出 $\eta_{max}$，$\cos\pi=-1$ 给出 $\eta_{min}$，中间平滑过渡。$\eta_{min}$ 通常不取 0——LLM 常设 $\eta_{min}=\eta_{max}/10$，防止衰减到底后彻底丧失学习能力。

**③ 指数 / step 衰减**。指数：$\eta_t=\eta_0\gamma^t$（$\gamma<1$，本文取 0.998）；step：$\eta_t=\eta_0\cdot0.1^{\lfloor t/T\rfloor}$。两者都是「每过固定步数乘一个因子」，实现简单、适合中小模型快速调参，但缺少 cosine 的平滑性。

## §1.5 手算例子①：warmup 前 10 步的学习率

设 $\eta_{max}=10^{-3}$、$T_w=10$，按 $\eta_t=\eta_{max}\cdot t/T_w$ 逐项手算（以 $10^{-4}$ 为单位）：

| step $t$ | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | 10 |
|---|---|---|---|---|---|---|---|---|---|---|----|
| $t/T_w$ | 0 | 0.1 | 0.2 | 0.3 | 0.4 | 0.5 | 0.6 | 0.7 | 0.8 | 0.9 | 1.0 |
| $\eta_t$ | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | 10 |

可见 warmup 阶段 lr 以固定等差 $10^{-4}$ 增长。若 $T_w$ 改为 1000，每步增量只剩 $10^{-6}$——**$T_w$ 决定爬升斜率**：步数越少、斜率越陡，前几步越容易冲出问题，这正是 §1.7 讲的步数敏感性。

下面用代码逐项核对本表，并顺带打印 cosine 在 100 步调度里的取值。

In [ ]:
# ---------- 手算例子验证：warmup 10 步表 + cosine 共 100 步取值 ----------
eta_max, Tw = 1e-3, 10
warm = [eta_max * t / Tw for t in range(Tw + 1)]
print('warmup 前 11 步 (×1e-4):', [round(v * 1e4, 4) for v in warm])

def cosine100(t):
    return eta_max * 0.5 * (1 + np.cos(np.pi * t / 100))

for t in [0, 25, 50, 75, 100]:
    print(f'cosine t={t:3d} -> {cosine100(t):.4e}  ({cosine100(t)/eta_max:.5f} × peak)')
assert abs(cosine100(0) - eta_max) < 1e-12 and abs(cosine100(100)) < 1e-12
print('端点核对通过：t=0 为峰值、t=100 归零；t=50 恰为峰值一半')

## §1.5（续）手算例子②：cosine 在 100 步内的取值

设总步数 $T=100$、峰值 $\eta_{max}=10^{-3}$、下限 $\eta_{min}=0$，相位 $\phi=\pi t/100$。用三个特殊角的余弦值手算：

| step $t$ | $\phi$ | $\cos\phi$ | $\eta_t$（×$10^{-3}$） |
|---|---|---|---|
| 0 | 0 | 1 | 1.0000（峰值） |
| 25 | $\pi/4$ | $\sqrt2/2\approx0.7071$ | 0.8536 |
| 50 | $\pi/2$ | 0 | 0.5000（半程） |
| 75 | $3\pi/4$ | $-\sqrt2/2$ | 0.1464 |
| 100 | $\pi$ | $-1$ | 0（归零） |

与上方代码输出逐项一致。两个要点：① **对称性**——$t=25\to50$ 与 $t=50\to75$ 各掉约 $0.354\times10^{-3}$，因为 $\cos$ 关于 $\phi=\pi/2$ 对称；② **瞬时斜率**——$d\eta/dt=-\tfrac{\eta_{max}}{2}\cdot\tfrac{\pi}{T}\sin\phi$ 在 $\phi=\pi/2$（$t=50$）处绝对值最大，所以**每步衰减速度在正中间最快、两端最慢**。

## §1.6 延伸：one-cycle 单周期策略

除了 warmup + cosine，还有 **one-cycle**：一个训练周期内，lr 先**线性升**到比常规峰值大得多的值，再**线性降**回很小值；动量反向调节（升期调小、降期调大）。Smith 与 Topin 2019 年提出，配合高 lr、小 batch、强数据增强可大幅加速训练（super-convergence）。

动机是「**大 lr 粗搜、小 lr 精修**」：前期用大 lr 快速穿越鞍点与浅谷、逃离不理想区域，后期小 lr 收尾。与 warmup + cosine 的区别：warmup 的爬升只是保护期、峰值才是常态；one-cycle 则把整个周期当成一次放大的探索-收敛过程。

工程提示：one-cycle 峰值常为常规 lr 的 3～10 倍，且需配合 BN 统计量的预热；对超大批量 LLM 预训练，warmup + cosine 仍是更稳、更可预期的选择。

In [ ]:
# ---------- 实验 2：不同调度下的收敛对比（简单二次函数 + 噪声） ----------
rng = np.random.default_rng(1)
a = 0.1
f_grad = lambda w: a * w + rng.normal(scale=0.05)

def run(sch, T=2000, Tw=200, peak=0.5):
    w = 3.0; dists = []
    for t in range(T):
        eta = schedule(sch, t, T, Tw, peak)
        w = w - eta * f_grad(w)
        dists.append(abs(w))
    return np.array(dists)

fig, ax = plt.subplots(figsize=(8, 4.0))
for sch, col, lab in [('warmup_cosine', '#4C72B0', 'warmup+cosine'),
                      ('step', '#55A868', 'step'),
                      ('constant', '#C44E52', 'constant')]:
    ax.plot(run(sch), label=lab, color=col, lw=1.4)
ax.set_xlabel('step'); ax.set_ylabel('|w|')
ax.set_title('带噪二次函数：调度帮助后期精细收敛', fontsize=12)
ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()
print('constant 后期在噪声下震荡；warmup+cosine 平滑收敛到更小 |w|')

### 实验 2 复盘：调度把「震荡」换成「收敛」

目标函数是 $f(w)=\tfrac12 aw^2$（梯度 $aw$）加上噪声，最优在 $w^*=0$。三条曲线对比：**constant（红）**在噪声下后期持续震荡，$|w|$ 停在一个可观的水平；**step（绿）**每 400 步缩小 lr，震荡幅度随之递减；**warmup + cosine（蓝）**衰减最平滑，最终 $|w|$ 最小。

这正是 §1.3 的定量版：常数 lr 时有效步长 $\eta\cdot a$ 与噪声尺度相当，参数在极小点附近做「布朗运动」；衰减后有效步长远小于噪声尺度，更新主要沿均值梯度方向推进。

值得记住的结论：**调度改善的是收敛精度与稳定性，而不是梯度信息本身**——同样的数据、同样的模型，只改 lr 曲线就能显著改变最终解。这是深度学习里性价比最高的超参数之一。

## §1.7 局限与补救

**① warmup 步数敏感**。$T_w$ 太短（几十步）时爬升过陡，前几步仍可能 spike；太长则浪费训练预算、峰值来得太晚。补救：设成总步数的 $1\%\sim5\%$；大批量训练按 batch 放大比例相应加长；观察前 100 步 loss 曲线，出现尖峰就加长 $T_w$。

**② cosine 与 early stopping 的矛盾**。cosine 把衰减预算摊满整个 $T$，若在 60\% 处 early stop，lr 还偏大、模型未收敛；若中途延长 $T$，曲线整体右移。补救：$T$ 定得略保守，或每次延长后重建调度器（PyTorch 的 `CosineAnnealingLR` 按传入的 `T_max` 计算，需同步更新）。

**③ 峰值与下限耦合**。峰值过高、$\eta_{min}$ 过低，后期有效步长过小、收敛变慢。补救：先固定 warmup 与峰值，单独扫描 $\eta_{min}/\eta_{max}\in[0.01,0.2]$，LLM 常用 0.1。

## §1.8 调度 × 优化器：SGD、Adam 与 AdamW

调度效果与优化器类型强相关。**SGD / 动量 SGD**：步长完全由 $\eta$ 决定，衰减最直接、收益最明显；**Adam**：每个维度自适应缩放（$\alpha/(\sqrt{\hat v}+\epsilon)$），对全局 lr 不那么敏感，但仍需要 warmup 与衰减——LLM 预训练清一色用 AdamW + cosine。

**AdamW 与 Adam 的区别**：权重衰减从梯度里解耦、直接乘到参数上。这使得 lr 调度只负责「更新步长」，权重衰减只负责「正则」，二者互不干扰，cosine 的收益更干净。

容易混淆的概念：**lr 衰减 ≠ 权重衰减**。前者随时间缩放更新幅度（影响收敛节奏），后者每步把参数按 $1-\lambda$ 拉向原点（影响解的位置与泛化）。面试若被问「Adam 还需要衰减吗」，回答：要——自适应只解决尺度，不解决「该不该越走越小」的时间维度。

## §2 梯度裁剪（gradient clipping）

**为什么**：RNN / LLM 的梯度范数可能随步长爆炸（连乘效应）→ 更新过大直接毁掉参数。

$$g \gets g \cdot \min\left(1,\;\frac{\text{max\_norm}}{\|g\|_2}\right)$$

只改方向长度、不改方向；`max_norm` 常用 1.0。PyTorch：`torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)`。


## §2.1 公式逐步推导：为什么方向不变

设梯度向量 $g$，L2 范数 $\|g\|_2=\sqrt{\sum_i g_i^2}$。裁剪目标：把范数压到上限 $c$ 以内，同时保持方向 $g/\|g\|_2$ 不变：
$$g'=\begin{cases}g,&\|g\|_2\le c\\c\cdot\dfrac{g}{\|g\|_2},&\|g\|_2>c\end{cases}$$
两段合写就是 PyTorch 公式：$g'=g\cdot\min(1,\ c/\|g\|_2)$。范数超限时，整个向量乘以同一个标量 $c/\|g\|_2<1$——每个分量同比例缩小，方向余弦不变。

**手算例子③**：$g=[10^4,\ -3\times10^3,\ 8\times10^3]$，则 $\|g\|_2=\sqrt{10^8+9\times10^6+64\times10^6}=\sqrt{1.73\times10^8}\approx1.315\times10^4$，远超 $c=1.0$：
$$g'\approx\frac{g}{1.315\times10^4}\approx[0.760,\ -0.228,\ 0.608],\qquad\|g'\|_2=1.0$$
实验 3 会精确复算这组数。注意：norm clip **只裁整体范数、不裁单个分量**——若个别分量异常大而整体范数正常，norm clip 不会动它，那是 value clip 的职责。

## §2.2 阈值怎么选：1.0 只是起点

`max_norm=1.0` 是最常被引用的经验值（PyTorch 文档示例、GPT 系列训练脚本常见），但它**不是普适真理**：不同模型、不同阶段，梯度范数分布可以相差几个数量级。更工程化的做法：

- 训练前几百步打印 $\|g\|_2$ 的均值 / 中位数 / 95\% 分位数，把阈值设在分位数量级，让裁剪**只在异常时介入**；
- RNN 与深 Transformer 早期范数大，常用 0.1～1.0；收敛后期范数变小，阈值只管上限、可保持不变；
- 区分两种裁剪：norm clip（`clip_grad_norm_`）保留整体方向、适合全局爆炸；value clip（`clip_grad_value_`）逐元素截断到 $[-c,c]$、适合个别分量离群。

**顺序铁律**：先 clip、后 `optimizer.step()`——clip 必须在 `backward()` 之后、更新之前调用，否则等于没裁。

In [ ]:
# ---------- 实验 3：梯度裁剪救爆炸 ----------
g_big = np.array([1e4, -3e3, 8e3])
g_clip = g_big * min(1.0, 1.0 / np.linalg.norm(g_big))
print(f'裁剪前 ||g||={np.linalg.norm(g_big):.3e} -> 裁剪后 ||g||={np.linalg.norm(g_clip):.3f}, 方向cos={np.dot(g_big,g_clip)/(np.linalg.norm(g_big)*np.linalg.norm(g_clip)):.6f}')
assert np.isclose(np.linalg.norm(g_clip), 1.0)
print('方向保持：裁剪只改模长不改方向')

## §3 梯度累积（gradient accumulation）

显存不够跑大 batch 时：小 batch 前向/反向 N 次，**梯度累加后再更新**，等价于 N 倍大 batch（权重更新一次）。

注意：BN / 统计量按小 batch 计算会与真大 batch 有差异；LayerNorm 不受影响（逐 token）→ LLM 常用。

In [ ]:
# ---------- 实验 4：梯度累积 vs 大 batch 等价性 ----------
rng = np.random.default_rng(2)
N, D = 128, 4
X = rng.normal(size=(N, D)); w_true = rng.normal(size=D)
y = X @ w_true + rng.normal(scale=0.1, size=N)

def g_batch(idx):
    r = X[idx] @ w0 - y[idx]
    return X[idx].T @ r / len(idx)

w0 = np.zeros(D)
g_full = g_batch(np.arange(N))            # 128 的全批梯度
g_acc = np.zeros(D)
for s in range(4):                        # 4 × 32 累积
    g_acc += g_batch(np.arange(s * 32, (s + 1) * 32)) / 4
print(f'全批梯度与累积梯度最大差: {np.abs(g_full - g_acc).max():.2e}')
assert np.abs(g_full - g_acc).max() < 1e-14
print('等价性验证通过：4×32 累积 = 1×128 全批（严格平均）')

## §3.1 梯度累积的三个隐蔽细节

**细节 ①：和还是均值？** 若每个小 batch 的梯度是「该 batch 的平均梯度」，累积 N 个后必须再除以 N（或每个先除以 N 再累加），否则总梯度是 N 倍均值、等效 lr 被放大 N 倍。实验 4 验证了严格平均下的等价性；框架里常见写法是 `loss.backward()` 累积梯度、每 N 次 `optimizer.step()` + `optimizer.zero_grad()`，靠「loss 除以 N」或「梯度除以 N」保证尺度正确。

**细节 ②：调度步数按什么计数？** warmup / cosine 里的 $t$ 指**优化器更新次数**（累积满 N 次后的 step），不是读过的样本数。设 $B_{global}=N\cdot B_{micro}$、总样本数 $D$，则更新步数 $=D/B_{global}$——调度曲线画在「更新步」这根横轴上。HF Trainer 默认按训练步数计数，配梯度累积时无需手动改，但要清楚横轴含义。

**细节 ③：与 BN 的冲突**。BN 统计量按微 batch 计算，N 次累积的口径与大 batch 不一致；LayerNorm 逐 token 归一、无此问题，所以 LLM 用梯度累积毫无负担。

## §4 EMA / SWA（参数滑动平均）

对**参数**做 EMA：$\theta_{ema} \gets \beta \theta_{ema} + (1-\beta)\theta$，推理用 EMA 参数。

作用：权重在多个局部极小间振荡时，EMA 取「质心」→ 更平坦、泛化更好（尤其配合 cosine 退火 = SWA）。


## §5 常见 bug 与面试追问

- **warmup 步数太短**：大模型至少数百步，否则 loss 早期 spike
- **cosine 忘留最小 lr**：降到底会破坏收敛，LLM 常设 `eta_min = peak * 0.1`
- **clip 与 lr 顺序**：先 clip 梯度，再优化器更新
- **梯度累积后忘除以 N**：累积的是和不是均值，步长被放大 N 倍


## §5.1 面试追问补充：三道高频题

**Q1：warmup 与 Adam 偏差校正是什么关系？** 偏差校正解决「$v_t$ 从 0 起步被低估」：$\hat v_t=v_t/(1-\beta_2^t)$。$\beta_2=0.999$ 时 $t=100$ 的校正因子约 $1/(1-0.999^{100})\approx10.5$——前 100 步 $v_t$ 仍被低估约一个数量级，不校正则 $\alpha/\sqrt{\hat v_t}$ 被放大；warmup 则是整体压低 $\eta$，从源头控制步长。二者互补：校正让尺度数学上正确，warmup 等统计量可信后再放开。所以「Adam 自带校正就不需要 warmup」是错的。

**Q2：clip 阈值为什么常见 1.0？** 梯度范数常被约束在 O(1) 量级（loss 取平均、初始化 O(1)），1.0 恰好是「不干预常规、只拦爆炸」的中间值；更严谨的做法是看范数分布取分位数（§2.2）。

**Q3：梯度累积后调度曲线会变吗？** 横轴是更新步而非数据步：等效 batch 变大 → 同一数据量下更新次数变少 → 每个更新步吃更多数据，衰减节奏变慢，warmup 步数也应按有效 batch 比例调整（§3.1 细节 ②）。

## 📝 公式小抄：三种调度一张表

| 调度 | 公式 | 关键参数 |
|---|---|---|
| 线性 warmup | $\eta_t=\eta_{max}\,t/T_w$ | $T_w$ 决定爬升斜率 |
| cosine | $\eta_t=\eta_{min}+\frac{\eta_{max}-\eta_{min}}{2}(1+\cos\frac{\pi(t-T_w)}{T-T_w})$ | $\eta_{min}$ 别取 0 |
| 指数 | $\eta_t=\eta_0\gamma^t$ | $\gamma\approx0.99\sim0.999$ |
| step | $\eta_t=\eta_0\,0.1^{\lfloor t/T\rfloor}$ | 突降有风险 |
| one-cycle | 线性升 + 线性降 | 峰值 3～10 倍 |

面试 30 秒速答：**warmup 管前期统计量冷启动，decay 管后期精细收敛**；gradient clipping 只缩范数不动方向；gradient accumulation 的横轴是更新步、别忘了除以 N；EMA 是参数层面的「质心」。

## §6 自测清单

- [ ] 画四种调度曲线（本文已出图）
- [ ] 默写 warmup 与 cosine 公式
- [ ] 默写梯度裁剪公式并解释方向不变
- [ ] 解释梯度累积为什么等价大 batch（严格平均时）
- [ ] 说清 EMA 为什么提升泛化

> 💡 下节预告：二阶方法（牛顿法/BFGS/K-FAC，06 篇），大模型内存实战（07 篇）。